# SHIPP Stage 5B — Silver Requests Development
Bronze bootcamp_students.shipp_bronze.lb_requests_history → Silver bootcamp_students.shipp_silver.silver_requests

Bronze CDC history (pinned Delta version)
  → keep insert + update_postimage + delete
  → newest event per request_id   (_pg_lsn DESC, _sort_by DESC)
  → drop keys whose newest event is delete
  → Silver contract + quality gate
  → overwrite Silver
  → reconcile Silver vs Bronze (two independent methods)
  → idempotency check

**Method:** Spark Window Current-State Reconstruction

```text
Bronze request history
    ↓
partition by request_id
    ↓
newest CDC event
    ↓
quality checks
    ↓
silver_requests
```

How to run
Close any AI assistant panel that has auto-approve on.
Run → Clear state and outputs, then Run all. Never run single cells out of order.
Every gate either prints PASS or stops with an AssertionError. A failure means the data is wrong — fix the cause, never loosen the assert.
Expected Silver content: every request whose newest Bronze event is not a delete — including demo-request-001.

This notebook is the only writer of silver_requests.


In [ ]:
%run ../common/shipp_silver_lib

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SHIPP — notebook bootstrap (do not run directly)
Every stage notebook starts with:

%run ../common/shipp_silver_lib
This cell finds the repo root, reloads the Python modules under config/ and data_pipeline/ (so an edited module is always picked up, even without Clear state), and exposes their functions with the notebook's spark session already bound.

All logic lives in .py files (data_pipeline/…), covered by tests/unit in CI. Notebooks only read, call those functions, run gates and write.

SHIPP bootstrap loaded from /Workspace/Users/raneem99hamarneh@yahoo.com/Shipp/shipp-marketplace-v3

# Config

In [ ]:
CATALOG = "bootcamp_students"

SOURCE_TABLE = f"{CATALOG}.shipp_bronze.lb_requests_history"
TARGET_SCHEMA = f"{CATALOG}.shipp_silver"
TARGET_TABLE = f"{TARGET_SCHEMA}.silver_requests"
BUSINESS_KEY = "request_id"

BRONZE_SCHEMA = "shipp_bronze"
SILVER_SCHEMA = "shipp_silver"

# NOTE: these request statuses follow the spec (Open → MatchesAvailable → ItemSaved → Closed).
# If the status gate fails, the printed 'Distinct statuses' shows what Lakebase really stores:
# update this set to match the Lakebase CHECK constraint — do NOT delete the check.
VALID_STATUSES = ["OPEN", "MATCHES_AVAILABLE", "ITEM_SAVED", "CLOSED"]

REQUIRED_SOURCE_COLUMNS = {
    "request_id",
    "requester_id",
    "request_text",
    "category",
    "location",
    "latitude",
    "longitude",
    "need_by_date",
    "status",
    "updated_at",
    "_pg_change_type",
    "_pg_lsn",
    "_sort_by",
}

EXPECTED_SILVER_COLUMNS = [
    "request_id",
    "requester_id",
    "request_text",
    "category",
    "latitude",
    "longitude",
    "need_by_date",
    "status",
    "updated_at",
    "source_pg_lsn",
    "source_sort_by",
    "source_change_type",
    "silver_processed_at",
]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET_SCHEMA}")

print("SOURCE_TABLE =", SOURCE_TABLE)
print("TARGET_TABLE =", TARGET_TABLE)
print("BUSINESS_KEY =", BUSINESS_KEY)

# Load Bronze (pinned) + source Schema contract

In [ ]:
bronze_df, bronze_version = read_bronze_pinned(SOURCE_TABLE)

print("Bronze row count:", bronze_df.count())

missing_columns = REQUIRED_SOURCE_COLUMNS - set(bronze_df.columns)
assert not missing_columns, f"Bronze schema contract failed. Missing: {sorted(missing_columns)}"
print("PASS — Bronze source schema contract.")

display(bronze_df.groupBy("_pg_change_type").count().orderBy("_pg_change_type"))

#  Delete-aware current-state reconstruction

In [ ]:
latest_events_df, current_rows_df, deleted_keys_df = reconstruct_current_state(bronze_df, BUSINESS_KEY)

print("Keys with any event:        ", latest_events_df.count())
print("Newest event = delete (out):", deleted_keys_df.count())
print("Current rows for Silver:    ", current_rows_df.count())

display(
    latest_events_df
    .select(BUSINESS_KEY, "_pg_change_type", "_pg_lsn", "_sort_by", "status")
    .orderBy(BUSINESS_KEY)
)

In [ ]:
display(
    bronze.groupBy("_pg_change_type")
    .count()
    .orderBy("_pg_change_type")
)


In [ ]:
latest_window = Window.partitionBy("request_id").orderBy(
    F.col("_pg_lsn").desc(),
    F.col("_sort_by").desc(),
)

ranked = bronze.withColumn(
    "_current_rn",
    F.row_number().over(latest_window),
)

current_requests = (
    ranked
    .filter(F.col("_current_rn") == 1)
    .filter(F.col("_pg_change_type").isin("insert", "update_postimage"))
    .filter(F.col("request_id").isNotNull())
    .filter(F.col("latitude").between(-90, 90))
    .filter(F.col("longitude").between(-180, 180))
    .select(
        "request_id",
        "requester_id",
        "request_text",
        "category",
        "status",
        "latitude",
        "longitude",
        "need_by_date",
        "updated_at",
        F.col("_pg_lsn").alias("source_pg_lsn"),
        F.col("_sort_by").alias("source_sort_by"),
        F.col("_pg_change_type").alias("source_change_type"),
        F.current_timestamp().alias("silver_processed_at"),
    )
)

display(current_requests)


In [ ]:
duplicate_keys = (
    current_requests.groupBy("request_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)
null_keys = current_requests.filter(F.col("request_id").isNull()).count()
invalid_coords = current_requests.filter(
    ~F.col("latitude").between(-90, 90)
    | ~F.col("longitude").between(-180, 180)
).count()

print("current rows:", current_requests.count())
print("duplicate request_id:", duplicate_keys)
print("null request_id:", null_keys)
print("invalid coordinates:", invalid_coords)

assert duplicate_keys == 0
assert null_keys == 0
assert invalid_coords == 0


In [ ]:
display(
    current_requests.filter(
        F.col("request_id") == "demo-request-001"
    )
)


In [ ]:
(
    current_requests.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET)
)

print("Wrote:", TARGET)
print("Silver request rows:", spark.table(TARGET).count())
